# DPM Engine · Request Dispatcher

**Purpose.** Runs the calculations requested with the DataLoader. It is run by the batch `DPM_Exec_Request`, which the DataLoader of the request file triggers when its load ends.

- Each row of the request file is one run of `DE_ENR_NB_LGL_REG_DPM_PROCESSING`: `reference_period` (mandatory), `frameworks_csv` (empty = all the frameworks of the parameterisation) and `metadata_as_of` (empty = `latest`).
- Runs the requests that have no run yet, oldest first, in the same Spark session. When it finishes, it looks for requests uploaded meanwhile, because the orchestrator skips the triggers that arrive while the batch is running. Without pending requests it ends without doing anything.
- Each request is tried once. Its run is recorded by `DE_ENR_NB_LGL_REG_DPM_PROCESSING` in `dp_lgl_reg_dpm_exec` with the identifier of the request (`req_id`), also when it fails; to try again, the request is uploaded again.
- A failed run does not stop the other requests; at the end the notebook ends with an error if any failed (`fail_on_failed_request`).

**Sources:** `raw_dpm_exec_req` (DataLoader, append), `dp_lgl_reg_dpm_exec`.

**Sink tables:** none directly (the runs write the tables of the engine).

**Parameters:** table names, `processing_notebook`, `processing_timeout_seconds`, `fail_on_failed_request`. The batch passes none.

**Output:** the notebook ends with the result of each request as its exit value (JSON).

**Location:** workspace *DP FET Legal and Regulatory Reporting*, folder `01_Processes/DPM`, default Lakehouse `DP_FET_LH_LegalAndRegulatoryReporting`.

**Changelog:** 2026-11-11 | Notebook creation.


## Session configuration

The runs share this session, so it carries the configuration of `DE_ENR_NB_LGL_REG_DPM_PROCESSING`.


In [1]:
%%configure -f
{
    "driverMemory": "7g",
    "driverCores": 2,
    "executorMemory": "7g",
    "executorCores": 2,
    "conf": {
        "spark.native.enabled": "true",
        "spark.dynamicAllocation.maxExecutors": "1"
    },
    "sessionTimeoutInSeconds": 7200
}

StatementMeta(, e8c8680e-90f6-4b88-92b4-000a2bd1314e, -1, Finished, Available, Finished, True)

<div style="font-family:monospace;font-size:13px">
    <p>
        <span style="color:red;font-weight:bold">Warning:</span> When you want to configure the driverMemory, driverCores, executorCores,
        executorMemory, or numExecutors, check the workspace settings and default pool node details from workspace settings -> Default pool
        for workspace -> Pool details, you can use %%configure set customized compute configurations that within the range of Pool level size.
    </p>
    <p>
        Notice that if the session configuration is different from the live session (8 vCores, 56GB memory, 2 executors), the session
        start will fall back to the on-demand session which will take about 3~5 mins to spin up.
    </p>
    <p>The Executor/Driver size buckets on Fabric are: </p>
    <ul>
        <li>A small size bucket: 4 vCores/28 GB memory</li>
        <li>A medium size bucket: 8 vCores/56 GB memory</li>
        <li>A large size bucket: 16 vCores/112 GB memory</li>
        <li>A xlarge size bucket: 32 vCores/224 GB memory</li>
        <li>A xxlarge size bucket: 64 vCores/400 GB memory</li>
    </ul>
    <p>
        We recommend avoiding using Driver or Executor vCore/Memory counts that are not in the size bucket specified above, so be sure
        to specify the <b>Driver and Executor</b> vCore/Memory values in relation to the above size buckets (4vCores/28GB, 8vCores/56GB,
        16vCores/112GB, 32vCores/224GB, or 64vCores/400GB), moreover, the executor size and driver size should be the same.
    </p>
</div>

## 1. Parameters


In [2]:
# Raw table loaded by the DataLoader with the execution requests (one row per run).
raw_requests_table = "raw_dpm_exec_req"
# Run table of the engine: a request is pending while it has no run.
run_execution_table = "dp_lgl_reg_dpm_exec"

# Notebook that calculates each request, and the maximum time of one run.
processing_notebook = "DE_ENR_NB_LGL_REG_DPM_PROCESSING"
processing_timeout_seconds = 7200

# True = the notebook ends with an error when a request fails, after trying them all, so
# that the batch shows it.
fail_on_failed_request = True


StatementMeta(, e8c8680e-90f6-4b88-92b4-000a2bd1314e, 5, Finished, Available, Finished, False)

## Common definitions

Helpers shared with the other notebooks of the engine (`DE_ENR_NB_LGL_REG_DPM_COMMON`).


In [3]:
%run DE_ENR_NB_LGL_REG_DPM_COMMON

StatementMeta(, e8c8680e-90f6-4b88-92b4-000a2bd1314e, 6, Finished, Available, Finished, True)

## 2. Requests and runs


In [4]:
import json

from notebookutils import mssparkutils

# Columns of the request file: the parameters of DE_ENR_NB_LGL_REG_DPM_PROCESSING.
REQUEST_COLUMNS = ("reference_period", "frameworks_csv", "metadata_as_of")


def pending_requests():
    """Requests that have not been run yet, oldest first.

    A request is a row of the request file with some value. Its identifier is the
    dataload_id of the upload plus a hash of its values, so the same row loaded twice in
    one upload is one request, and the same values in a later upload are a new one. A
    request is pending while no run of the run table carries its identifier: each request
    is tried once, whatever the result of its run.
    """
    values = [F.coalesce(F.trim(F.col(name)), F.lit("")).alias(name) for name in REQUEST_COLUMNS]
    requests = (
        spark.table(raw_requests_table)
        .select(*values, "dataload_id", "ingestion_timestamp", "usr_id")
        .where(F.length(F.concat(*REQUEST_COLUMNS)) > 0)
        .withColumn(
            "req_id",
            F.concat_ws(
                "-", "dataload_id", F.substring(F.sha2(F.concat_ws("|", *REQUEST_COLUMNS), 256), 1, 12)
            ),
        )
        .groupBy("req_id", *REQUEST_COLUMNS)
        .agg(F.min("ingestion_timestamp").alias("request_dt"), F.max("usr_id").alias("usr_id"))
    )
    done = spark.table(run_execution_table).where(F.col("req_id").isNotNull()).select("req_id").distinct()
    pending = requests.join(done, "req_id", "left_anti").orderBy("request_dt", "reference_period", "req_id")
    return [row.asDict() for row in pending.toLocalIterator()]


def run_request(request):
    """Runs DE_ENR_NB_LGL_REG_DPM_PROCESSING for one request; returns its result.

    The run records itself, with the identifier of the request, in the run table and in
    the issue log. A run that ends with an error does not stop the other requests.
    """
    result = {
        "req_id": request["req_id"],
        "reference_period": request["reference_period"],
        "frameworks_csv": request["frameworks_csv"],
        "usr_id": clean(request["usr_id"]),
    }
    try:
        summary = json.loads(
            mssparkutils.notebook.run(
                processing_notebook,
                int(processing_timeout_seconds),
                {
                    "reference_period": request["reference_period"],
                    "frameworks_csv": request["frameworks_csv"],
                    "metadata_as_of": request["metadata_as_of"] or "latest",
                    "req_id": request["req_id"],
                },
            )
        )
        result.update(exec_id=summary["exec_id"], exec_stts=summary["exec_stts"])
    except Exception as error:  # the detail of the failure is in the run table and the issue log
        result.update(exec_id="", exec_stts="FAILED", msg_ln=str(error)[:500])
    return result


def run_dispatcher():
    """Runs the pending requests one after another, in the same session; returns their results.

    The orchestrator skips a trigger of the batch while the batch is running, so the
    requests uploaded during this run would not be run by anyone: when the run finishes
    its requests it looks again, until there are no new ones. A request is tried once per
    run of the notebook. Without pending requests it ends without doing anything.
    """
    results, tried = [], set()
    while True:
        requests = [request for request in pending_requests() if request["req_id"] not in tried]
        if not requests:
            break
        for request in requests:
            tried.add(request["req_id"])
            results.append(run_request(request))
    print(json.dumps(results, indent=2))
    failed = [result["req_id"] for result in results if result["exec_stts"] == "FAILED"]
    if failed and bool(fail_on_failed_request):
        raise RuntimeError(f"Request(s) with a failed run: {failed}. See the run table and the issue log.")
    return results


StatementMeta(, e8c8680e-90f6-4b88-92b4-000a2bd1314e, 7, Finished, Available, Finished, False)

## 3. Execute requests


In [7]:
dispatcher_result = run_dispatcher()
# Exit value for the batch that launches this notebook (result of each request, as JSON).
mssparkutils.notebook.exit(json.dumps(dispatcher_result))


StatementMeta(, e8c8680e-90f6-4b88-92b4-000a2bd1314e, 10, Finished, Available, Finished, False)

[
  {
    "req_id": "1275-3523a03fb65b",
    "reference_period": "202609",
    "frameworks_csv": "FINREP",
    "usr_id": "laura.martinez@novobanco.pt",
    "exec_id": "FINREP_202609_20261117T125020",
    "exec_stts": "SUCCESS_WITH_WARNINGS"
  }
]
ExitValue: [{"req_id": "1275-3523a03fb65b", "reference_period": "202609", "frameworks_csv": "FINREP", "usr_id": "laura.martinez@novobanco.pt", "exec_id": "FINREP_202609_20261117T125020", "exec_stts": "SUCCESS_WITH_WARNINGS"}]